# TRIAGE-EG Stage 0 BTC Data Audit

**This notebook runs Stage 0 BTC Data Audit, not retrieval or frame extraction.**

BTC-only, dataset read-only; output nằm dưới `/kaggle/working`. Không chạy Fast Line, Event Graph, Agent hoặc model.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
DATASET_ROOT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
OUTPUT_ROOT = Path(os.environ.get('AIC_AUDIT_OUTPUT_ROOT', '/kaggle/working/triage_eg_stage0_audit'))
AUDIT_MODE = os.environ.get('AIC_AUDIT_MODE', 'sample').lower()
RESUME = os.environ.get('AIC_AUDIT_RESUME', '0') == '1'
print('Python:', sys.version)
print('repository ref:', REPO_REF)
print('dataset root:', DATASET_ROOT)
print('output root:', OUTPUT_ROOT)
print('audit mode:', AUDIT_MODE)

In [ ]:
def git(*args, cwd=None):
    completed = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True, check=False)
    if completed.returncode:
        raise RuntimeError(completed.stderr.strip() or completed.stdout.strip())
    return completed.stdout.strip()

if (REPO_DIR / '.git').is_dir():
    if REFRESH_REPO:
        git('fetch', '--depth', '1', 'origin', REPO_REF, cwd=REPO_DIR)
        git('checkout', '--detach', 'FETCH_HEAD', cwd=REPO_DIR)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout')
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git('clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_DIR))
    git('fetch', '--depth', '1', 'origin', REPO_REF, cwd=REPO_DIR)
    git('checkout', '--detach', 'FETCH_HEAD', cwd=REPO_DIR)
COMMIT = git('rev-parse', 'HEAD', cwd=REPO_DIR)
print('resolved commit:', COMMIT)

In [ ]:
module_file = REPO_DIR / 'src/triage_eg/data/stage0_audit/runner.py'
if not module_file.is_file():
    raise RuntimeError(f'Missing {module_file}. Ensure Stage 0 is committed to {REPO_REF}; set AIC_REFRESH_REPO=1 and rerun.')
if not DATASET_ROOT.is_dir(): raise RuntimeError(f'Dataset root missing: {DATASET_ROOT}')
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
probe = shutil.which('ffprobe')
print('ffprobe:', probe or 'NOT AVAILABLE (raw-video gate will fail; BTC audit continues)')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('preflight: PASS')

In [ ]:
from triage_eg.data.stage0_audit import AuditConfig, run_audit
SAMPLE_SIZE = int(os.environ.get('AIC_AUDIT_SAMPLE_SIZE', '10'))
CLIP_VALIDATION = os.environ.get('AIC_AUDIT_CLIP_VALIDATION', 'full')
OBJECT_VALIDATION = os.environ.get('AIC_AUDIT_OBJECT_VALIDATION', 'full')
FFPROBE_TIMEOUT = int(os.environ.get('AIC_AUDIT_FFPROBE_TIMEOUT', '30'))
if AUDIT_MODE not in {'sample','full'}: raise ValueError('AIC_AUDIT_MODE must be sample or full')
if OUTPUT_ROOT.joinpath('checkpoints').is_dir() and os.environ.get('AIC_AUDIT_RESUME') is None:
    RESUME = True
OVERWRITE = not RESUME
config = AuditConfig(dataset_root=DATASET_ROOT, output_root=OUTPUT_ROOT, mode=AUDIT_MODE, sample_size=SAMPLE_SIZE, seed=2026, clip_validation=CLIP_VALIDATION, object_validation=OBJECT_VALIDATION, max_object_json_bytes=1048576, ffprobe_timeout_seconds=FFPROBE_TIMEOUT, resume=RESUME, overwrite=OVERWRITE, strict_root=True)
print({'mode':AUDIT_MODE,'sample_size':SAMPLE_SIZE,'clip':CLIP_VALIDATION,'object':OBJECT_VALIDATION,'resume':RESUME,'overwrite':OVERWRITE,'ffprobe_timeout':FFPROBE_TIMEOUT})

In [ ]:
started = time.monotonic()
audit_result = run_audit(config, project_root=REPO_DIR)
elapsed = time.monotonic() - started
print('This notebook runs Stage 0 BTC Data Audit, not retrieval or frame extraction.')

In [ ]:
summary = audit_result.summary
print({k:summary[k] for k in ('videos_selected','videos_completed','videos_resumed','videos_failed')})
print('elapsed_seconds:', elapsed)
for name,path in audit_result.paths.items(): print(name, '->', path)

In [ ]:
print({k:summary[k] for k in ('mapping_rows','keyframe_files','clip_rows','object_files','detections_observed','metadata_files')})
print('issue severity:', summary['issues']['by_severity'])
print('top issue codes:', sorted(summary['issues']['by_code'].items(), key=lambda item:(-item[1],item[0]))[:20])

In [ ]:
print('BTC baseline gate:', summary['gates']['btc_baseline'])
print('Raw video gate:', summary['gates']['raw_video'])

In [ ]:
issues_path = audit_result.paths['audit_issues.jsonl']
sample_issues=[]
with issues_path.open(encoding='utf-8') as stream:
    for line in stream:
        if line.strip(): sample_issues.append(json.loads(line))
        if len(sample_issues) >= 30: break
for item in sample_issues: print({k:item.get(k) for k in ('severity','code','video_id','ordinal_n','asset_type','message')})

In [ ]:
notes=json.loads(audit_result.paths['contract_notes.json'].read_text(encoding='utf-8'))
for key in ('btc_ordinal_contract','original_frame_policy','duplicate_frame_idx_policy','numeric_string_policy','bbox_order','clip_model_compatibility'): print(key, ':', notes[key])

In [ ]:
from triage_eg.data.stage0_audit.writers import FINAL_ARTIFACTS, create_bundle
from zipfile import ZipFile
for name in FINAL_ARTIFACTS:
    if not (OUTPUT_ROOT/name).is_file(): raise RuntimeError(f'Missing final artifact: {name}')
zip_path = Path('/kaggle/working/triage_eg_stage0_audit_bundle.zip')
create_bundle(OUTPUT_ROOT, zip_path)
with ZipFile(zip_path) as archive: members=archive.namelist()
assert members == list(FINAL_ARTIFACTS)
assert not any(name.startswith(('checkpoints/','logs/')) for name in members)
assert zip_path.name not in members
print('DOWNLOAD ZIP:', zip_path)
print('size_bytes:', zip_path.stat().st_size)
print('members:', members)